# Assignment 3 - NYC Taxi Big Data Processing

In [1]:
import dask
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
import dask.dataframe as dd
from pathlib import Path
import os
import pandas as pd

## Cluster setup (SLURM + Dask)

In [2]:
# Verify that Dask is installed
print("Dask version:", dask.__version__)

Dask version: 2024.5.0


In [3]:
cluster = SLURMCluster(
    queue="all",
    cores=1, # lower number of cores per worker to reduce memory usage
    memory="16GB",
    walltime="02:00:00",
    job_extra=["--output=/d/hpc/home/sm79111/bigdata/logs/slurm-%j.out"]
)

cluster.scale(8)  # Scale to 8 workers

client = Client(cluster)
client

/d/hpc/home/sm79111/.conda/envs/bd39/lib/python3.9/site-packages/dask_jobqueue/core.py:266: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/d/hpc/home/sm79111/.conda/envs/bd39/lib/python3.9/site-packages/dask_jobqueue/core.py:266: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)


Connection method: Cluster object,Cluster type: dask_jobqueue.SLURMCluster
Dashboard: http://153.5.72.244:8787/status,
Dashboard: http://153.5.72.244:8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://153.5.72.244:37051,Workers: 0
Dashboard: http://153.5.72.244:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


## Dataset Overview
We focus on the **yellow taxi dataset**, which is the largest and most commonly used.

In [4]:
DATA_DIR = Path("/d/hpc/projects/FRI/bigdata/data/Taxi")

files = sorted(DATA_DIR.rglob("yellow_tripdata_*.parquet"))

print("Total yellow taxi parquet files:", len(files))

for f in files[:10]:
    print(f.name)

Total yellow taxi parquet files: 193
yellow_tripdata_2009-01.parquet
yellow_tripdata_2009-02.parquet
yellow_tripdata_2009-03.parquet
yellow_tripdata_2009-04.parquet
yellow_tripdata_2009-05.parquet
yellow_tripdata_2009-06.parquet
yellow_tripdata_2009-07.parquet
yellow_tripdata_2009-08.parquet
yellow_tripdata_2009-09.parquet
yellow_tripdata_2009-10.parquet


### Dataset Organization by Year
To simplify processing we group the parquet files by year.
Each year contains monthly files.

In [5]:
files_by_year = {}

for f in files:
    year = f.stem.split("_")[-1][:4]
    files_by_year.setdefault(year, []).append(str(f))

print("Available years:")
print(sorted(files_by_year.keys()))

Available years:
['2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


### Schema Inspection and Harmonization
Taxi datasets evolve over time.

Column names and data types may change between years. Before combining datasets we inspect schemas to detect inconsistencies.

In [6]:
selected_years = ["2019", "2020", "2021", "2022", "2023"]

for year in selected_years:
    df = dd.read_parquet(files_by_year[year])
    
    print("\n===== YEAR", year, "=====")
    print("Number of columns:", len(df.columns))
    print(df.dtypes.astype(str))


===== YEAR 2019 =====
Number of columns: 19
VendorID                          int64
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               string
PULocationID                      int64
DOLocationID                      int64
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
airport_fee                      object
dtype: object

===== YEAR 2020 =====
Number of columns: 19
VendorID                          int64
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count 

In [7]:
schemas = {}

for year in selected_years:
    df = dd.read_parquet(files_by_year[year])
    schemas[year] = set(df.columns)

for year, cols in schemas.items():
    print(year, len(cols))

2019 19
2020 19
2021 19
2022 19
2023 19


## Data Ingestion Tasks

1. CSV for 5 years
2. CSV + HDF5 for 1 year
3. Optimized Parquet partitions

In [8]:
OUTPUT_DIR = Path("/d/hpc/home/sm79111/bigdata/output")

CSV_DIR = OUTPUT_DIR / "csv"
HDF_DIR = OUTPUT_DIR / "hdf"
PARQUET_DIR = OUTPUT_DIR / "parquet"

CSV_DIR.mkdir(parents=True, exist_ok=True)
HDF_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_DIR.mkdir(parents=True, exist_ok=True)

### Schema Normalization

Taxi datasets evolve over time. Some columns appear only in later years and data types may change. To ensure consistency across datasets we define a
target schema and normalize each partition accordingly.

In [9]:
target_columns = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "store_and_fwd_flag",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "airport_fee"
]

def normalize_partition(pdf):

    pdf = pdf.copy()

    # add missing columns
    for col in target_columns:
        if col not in pdf.columns:
            pdf[col] = pd.NA

    # reorder columns
    pdf = pdf[target_columns]

    numeric_cols = [
        "VendorID","passenger_count","trip_distance","RatecodeID",
        "PULocationID","DOLocationID","payment_type",
        "fare_amount","extra","mta_tax","tip_amount",
        "tolls_amount","improvement_surcharge","total_amount",
        "congestion_surcharge","airport_fee"
    ]

    for col in numeric_cols:
        pdf[col] = pd.to_numeric(pdf[col], errors="coerce")

    pdf["store_and_fwd_flag"] = pdf["store_and_fwd_flag"].astype("string")

    pdf["tpep_pickup_datetime"] = pd.to_datetime(
        pdf["tpep_pickup_datetime"], errors="coerce"
    )

    pdf["tpep_dropoff_datetime"] = pd.to_datetime(
        pdf["tpep_dropoff_datetime"], errors="coerce"
    )

    return pdf

In [10]:
# Define metadata 
meta = pd.DataFrame({
    "VendorID": pd.Series(dtype="float64"),
    "tpep_pickup_datetime": pd.Series(dtype="datetime64[ns]"),
    "tpep_dropoff_datetime": pd.Series(dtype="datetime64[ns]"),
    "passenger_count": pd.Series(dtype="float64"),
    "trip_distance": pd.Series(dtype="float64"),
    "RatecodeID": pd.Series(dtype="float64"),
    "store_and_fwd_flag": pd.Series(dtype="string"),
    "PULocationID": pd.Series(dtype="float64"),
    "DOLocationID": pd.Series(dtype="float64"),
    "payment_type": pd.Series(dtype="float64"),
    "fare_amount": pd.Series(dtype="float64"),
    "extra": pd.Series(dtype="float64"),
    "mta_tax": pd.Series(dtype="float64"),
    "tip_amount": pd.Series(dtype="float64"),
    "tolls_amount": pd.Series(dtype="float64"),
    "improvement_surcharge": pd.Series(dtype="float64"),
    "total_amount": pd.Series(dtype="float64"),
    "congestion_surcharge": pd.Series(dtype="float64"),
    "airport_fee": pd.Series(dtype="float64"),
    "year": pd.Series(dtype="int64")
})

# Normalize data for selected years and store in a dictionary
normalized_dfs = {}

for year in selected_years:

    print("Processing year:", year)

    yearly_parts = []

    # Read and normalize each parquet file separately to handle schema drift
    # across monthly files (e.g., airport_fee missing in some 2023 files).
    for file_path in files_by_year[year]:
        part_df = dd.read_parquet(
            file_path,
            engine="pyarrow"
        )

        for col in target_columns:
            if col not in part_df.columns:
                part_df[col] = None

        part_df = part_df.map_partitions(
            normalize_partition,
            meta=meta.drop(columns=["year"])
        )

        yearly_parts.append(part_df)

    df = dd.concat(yearly_parts, interleave_partitions=True)
    df = df.assign(year=int(year))

    # Persist the normalized DataFrame to avoid re-computation in later steps
    normalized_dfs[year] = df.persist()

Processing year: 2019
Processing year: 2020
Processing year: 2021
Processing year: 2022
Processing year: 2023


### Export Five Years of Data to CSV
CSV is a common but inefficient storage format. We export five years of
data to CSV for comparison with other formats.

In [13]:
for year in selected_years:

    out_dir = CSV_DIR / f"csv_{year}"
    out_dir.mkdir(parents=True, exist_ok=True)

    df = normalized_dfs[year]

    print(year, "partitions:", df.npartitions)

    df.to_csv(
        str(out_dir / "part-*.csv"),
        index=False,
        compute_kwargs={"retries": 2}
    )

    print(f"{year} CSV exported")

2019 partitions: 12


2019 CSV exported
2020 partitions: 12
2020 CSV exported
2021 partitions: 12
2021 CSV exported
2022 partitions: 12
2022 CSV exported
2023 partitions: 12
2023 CSV exported


### Export for 1 year in both _HDF5_ and _CSV_

In [11]:
one_year = "2023"
one_year_df = normalized_dfs[one_year].persist()

one_year_csv_dir = OUTPUT_DIR / f"one_year_csv_{one_year}"
one_year_csv_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Export CSV
one_year_df.to_csv(
    str(one_year_csv_dir / "part-*.csv"),
    index=False
)

In [12]:
df_2023 = dd.read_csv(str(one_year_csv_dir / "part-*.csv"), assume_missing=True)

for i in range(df_2023.npartitions):
    df_2023.get_partition(i).compute().to_hdf(
        HDF_DIR / "2023.h5",
        key="taxi",
        mode="a",
        format="table",
        append=True
    )

In [13]:
pd.read_hdf(HDF_DIR / "2023.h5", key="taxi").head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,year
0,2.0,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161.0,141.0,2.0,9.3,1.00,0.5,0.00,0.0,1.0,14.30,2.5,0.00,2023.0
1,2.0,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43.0,237.0,1.0,7.9,1.00,0.5,4.00,0.0,1.0,16.90,2.5,0.00,2023.0
2,2.0,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48.0,238.0,1.0,14.9,1.00,0.5,15.00,0.0,1.0,34.90,2.5,0.00,2023.0
3,1.0,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,1.0,N,138.0,7.0,1.0,12.1,7.25,0.5,0.00,0.0,1.0,20.85,0.0,1.25,2023.0
4,2.0,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,1.0,N,107.0,79.0,1.0,11.4,1.00,0.5,3.28,0.0,1.0,19.68,2.5,0.00,2023.0


### Optimized Parquet Creation

In [16]:
# Concatenate all years into a single Dask DataFrame for partitioned output
all_years_df = dd.concat([df for df in normalized_dfs.values()])

# Repartition by approximate size (256 MB per partition)
all_years_df = all_years_df.repartition(partition_size="256MB")

all_years_df.to_parquet(
    PARQUET_DIR / "optimized_parquet",
    engine="pyarrow",
    write_index=False,
    partition_on=["year"],
    row_group_size=100000,
    compute_kwargs={"retries": 2}
)

## Storage Size Comparison

In [17]:
def path_size_bytes(path: Path) -> int:
    path = Path(path)
    if path.is_file():
        return path.stat().st_size
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())

def fmt_gb(num_bytes: int) -> float:
    return num_bytes / (1024 ** 3)

In [ ]:
# 1) Compare the 5-year exports
original_parquet_size = sum(path_size_bytes(p) for year in selected_years for p in files_by_year[year])
csv_5yr_size = sum(path_size_bytes(CSV_DIR / f"csv_{year}") for year in selected_years)
parquet_5yr_size = path_size_bytes(PARQUET_DIR / "optimized_parquet")

five_year_compare = pd.DataFrame([
    {"format": "Original Parquet (2019-2023)", "size_gb": fmt_gb(original_parquet_size)},
    {"format": "CSV (2019-2023)", "size_gb": fmt_gb(csv_5yr_size)},
    {"format": "Parquet (2019-2023)", "size_gb": fmt_gb(parquet_5yr_size)},
])

five_year_compare["ratio_vs_csv"] = five_year_compare["size_gb"] / five_year_compare.loc[
    five_year_compare["format"] == "CSV (2019-2023)", "size_gb"
].iloc[0]

print("Five-year storage comparison:")
print(five_year_compare.sort_values("size_gb").to_string(index=False))

Five-year storage comparison:
                      format   size_gb  ratio_vs_csv
Original Parquet (2019-2023)  3.118586      0.142814
         Parquet (2019-2023)  4.005278      0.183419
             CSV (2019-2023) 21.836775      1.000000


In [21]:
# 2) Compare the 2023 dataset in all available formats
original_parquet_2023_size = sum(path_size_bytes(p) for p in files_by_year[one_year])
csv_2023_size = path_size_bytes(one_year_csv_dir)
hdf_2023_size = path_size_bytes(HDF_DIR / "2023.h5")
parquet_2023_size = path_size_bytes(PARQUET_DIR / "optimized_parquet" / f"year={one_year}")

one_year_compare = pd.DataFrame([
    {"format": "Original Parquet (2023)", "size_gb": fmt_gb(original_parquet_2023_size)},
    {"format": "CSV (2023)", "size_gb": fmt_gb(csv_2023_size)},
    {"format": "HDF5 (2023)", "size_gb": fmt_gb(hdf_2023_size)},
    {"format": "Parquet (2023)", "size_gb": fmt_gb(parquet_2023_size)},
])

one_year_compare["ratio_vs_csv"] = one_year_compare["size_gb"] / one_year_compare.loc[
    one_year_compare["format"] == "CSV (2023)", "size_gb"
].iloc[0]

print("\n2023 storage comparison:")
print(one_year_compare.sort_values("size_gb").to_string(index=False))


2023 storage comparison:
                 format  size_gb  ratio_vs_csv
Original Parquet (2023) 0.592082      0.154813
         Parquet (2023) 0.760043      0.198730
             CSV (2023) 3.824503      1.000000
            HDF5 (2023) 7.190747      1.880178


In [22]:
client.close()
cluster.close()

/d/hpc/home/sm79111/.conda/envs/bd39/lib/python3.9/site-packages/dask_jobqueue/core.py:266: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
